Basic settings

In [1]:
from utgc.runner import EnsembleRunner
runner = EnsembleRunner.from_config('output/SA/testrun/config.yaml')

Initializing runner from configuration: output/SA/testrun/config.yaml
Loaded 2745 segments from data/UT_vtds.geojson
Loaded 4 districts from maps/US-House/2025_USH_Leg-C/2025_USH_Leg-C.shp
Projecting to EPSG:26912
Found 1978 nodes assigned to 254 incorporated municipalities
Assigning unique IDs to unincorporated nodes...
Assigned unique IDs to 767 unincorporated nodes
Total unique MUNIIDs: 1021
  Graph built with 2745 nodes, 7641 edges
Population deviation tolerance: 0.100000%
Constraint: split max 4 muni
Constraint: max 0 multi-splits of muni
  Added locality split updater: 'ls_muni'
    Ignoring updater in output: 'ls_muni'
  Added split updater: 'split_muni'
  Added multi-split updater: 'muni_multi_splits'
Constraint: split max 5 county
Constraint: max 0 multi-splits of county
  Added locality split updater: 'ls_county'
    Ignoring updater in output: 'ls_county'
  Added split updater: 'split_county'
  Added multi-split updater: 'county_multi_splits'
Constraint: prevent same map fro

Election settings

In [2]:
runner = (runner
    .add_election_updaters(
        years=[2016, 2020, 2024],
        elections=['PRE','GOV','ATG','AUD','TRE'],
        parties_to_columns_override={
            "2024GOV":{"R1":"G24GOVRHEN","R2":"G24GOVNCLA"}
        }
    )
    .add_election_aggregator(
        "sb1011_data",
        ["2016PRE", "2016GOV", "2016ATG", "2016AUD", "2016TRE", "2020PRE", "2020GOV", "2020ATG", "2024PRE", "2024GOV", "2024ATG", "2024AUD", "2024TRE"],
        parties=["D", "R", "-"]
    )
    .add_election_metric_updaters(
        "sb1011_data",
        ["partisan_bias_utah", "partisan_bias", "mean_median", "efficiency_gap", "stdev_partisan_share", "majority_partisan_shares", "majority_seats"]
    )
)

  Added election updater: '2016ATG'
    Parties to columns: {'D': 'G16ATGDHAR', 'R': 'G16ATGRREY', '-': 'G16ATG-TOT'}
    Ignoring updater in output: '2016ATG'
  Added election updater: '2016AUD'
    Parties to columns: {'R': 'G16AUDRDOU', 'D': 'G16AUDDMIT', '-': 'G16AUD-TOT'}
    Ignoring updater in output: '2016AUD'
  Added election updater: '2016GOV'
    Parties to columns: {'D': 'G16GOVDWEI', 'R': 'G16GOVRHER', '-': 'G16GOV-TOT'}
    Ignoring updater in output: '2016GOV'
  Added election updater: '2016PRE'
    Parties to columns: {'D': 'G16PREDCLI', 'R': 'G16PRERTRU', '-': 'G16PRE-TOT'}
    Ignoring updater in output: '2016PRE'
  Added election updater: '2016TRE'
    Parties to columns: {'D': 'G16TREDHAN', 'R': 'G16TRERDAM', '-': 'G16TRE-TOT'}
    Ignoring updater in output: '2016TRE'
  Added election updater: '2020ATG'
    Parties to columns: {'R': 'G20ATGRREY', 'D': 'G20ATGDSKO', '-': 'G20ATG-TOT'}
    Ignoring updater in output: '2020ATG'
  Added election updater: '2020AUD'
    

In [3]:
# Set up visualization of sample maps
import os
import utgc.plotting as gcplot
import utgc.notebookhelper as nbh

munis, counties = nbh.load_boundaries_from_shapefiles()

runner = runner.add_runtime_callback(
    name="render_maps",
    frequency=10,
    action=lambda partition, step, output_dir: gcplot.visualize_partition(
        partition,
        step,
        os.path.join(output_dir, "maps"),
        counties=counties,
        municipalities=munis,
        split_munis_count=partition["split_muni"],
        split_counties_count=partition["split_county"],
    )
)

Loaded 29 counties from data/bounds/UtahCountyBoundaries/ut_cnty_2020_bound.shp
Loaded 259 municipalities from data/bounds/UtahMunicipalBoundaries/Municipalities.shp
Registered callback 'render_maps' (<lambda>) to run every 10 steps.


In [4]:
runner = runner.precondition(steps=50)
runner.run(
    name="ensemble",
    output_dir="output/SA/",
    num_steps=250,
)

=== Preconditioning ===
Starting preconditioning...


 38%|███▊      | 19/50 [00:10<00:17,  1.79it/s]


KeyboardInterrupt: 

In [ ]:
from utgc.results import ResultSet
import matplotlib.pyplot as plt
results = ResultSet(output_file="output/SA/ensemble/output.jsonl")

results.plot_metric_violin("majority_partisan_shares")